# Module 2: Dataframe Analysis — Homework

Stock Markets Analytics Zoomcamp 2026 (DataTalksClub).

Task: https://github.com/DataTalksClub/stock-markets-analytics-zoomcamp/blob/main/cohorts/2026/homework2.md

Run the cells top to bottom: Q3 reuses the `stocks_df` built in Q2.

In [1]:
!pip install -q yfinance gdown pyarrow lxml html5lib

In [2]:
import re
from io import StringIO

import numpy as np
import pandas as pd
import requests
import yfinance as yf

HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; zoomcamp-homework/1.0)"}


def fetch_tables(url):
    """Fetch a page with a User-Agent header and return its longest table.

    IPOScoop returns 403 for the default requests/pandas user agent, and the
    page contains several tables - the data one is always the longest.
    """
    html = requests.get(url, headers=HEADERS, timeout=60).text
    tables = pd.read_html(StringIO(html))
    df = max(tables, key=len)
    df.columns = [re.sub(r"\s+", " ", str(c)).strip() for c in df.columns]
    return df


def find_col(df, *parts):
    """Locate a column by substrings, so header line breaks do not matter."""
    for c in df.columns:
        if all(p.lower() in c.lower() for p in parts):
            return c
    raise KeyError(parts)


def to_num(x):
    """Parse '$8.00' -> 8.0, '1,234' -> 1234.0, '-' or blank -> NaN."""
    s = str(x).strip()
    if s in ("", "-", "--", "nan", "None", "N/A"):
        return np.nan
    s = s.replace("$", "").replace(",", "").strip()
    try:
        return float(s)
    except ValueError:
        return np.nan

## Q1. Withdrawn IPOs by company type

Load the Recently Filed table, keep rows where `Expected To Trade == 'Withdrawn'`
(32 expected), classify each company by name, derive an average price, compute the
offered value and aggregate by class.

The classification rules are **ordered** - the first match wins. That ordering changes
the answer: `Coolbit Technologies Ltd.` is `Technologies` rather than `Limited`, and
`pan-Africa Corp.` is `Acquisition Corp` because the rule matches the bare substring
`Corp`, not only the full `Acquisition Corp`.

In [3]:
df = fetch_tables("https://www.iposcoop.com/ipos-recently-filed/")
print(df.shape)
print(list(df.columns))
df.head(3)

(500, 10)
['File Date', 'Company', 'Symbol', 'Managers', 'Shares (millions)', 'Price Low', 'Price High', 'Est $ Vol (millions)', 'Expected To Trade', 'SCOOP Rating']


,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating
0,2026-09-14,ADARx Pharmaceuticals,ADRX,J.P.Morgan/Morgan Stanley/TD Cowen/UBS Investm...,0.00,NaN,NaN,$100.00,TBA,S/O
1,2026-09-14,Bamboo Insurance Services,BMB,J.P.Morgan/Morgan Stanley/Deutsche Bank Securi...,35.00,$18.00,$20.00,$665.00,Wednesday,S/O
2,2026-09-14,Electra Therapeutics,ETRA,Jefferies/TD Cowen/Evercore ISI/Cantor,21.67,$14.00,$16.00,$325.05,Friday,S/O


In [4]:
C_TRADE = find_col(df, "Expected", "Trade")
C_NAME = find_col(df, "Company")
C_SHARES = find_col(df, "Shares")
C_LOW = find_col(df, "Price", "Low")
C_HIGH = find_col(df, "Price", "High")
C_VOL = find_col(df, "Est", "Vol")

withdrawn = df[df[C_TRADE].astype(str).str.strip().str.lower() == "withdrawn"].copy()
print("withdrawn rows:", len(withdrawn))  # expected: 32

withdrawn rows: 32


In [5]:
def company_type(name):
    """Ordered classification - the first matching rule wins."""
    n = str(name)
    if "Technologies" in n:
        return "Technologies"
    if "Acquisition Corp" in n or "Acquisition Corporation" in n or "Corp" in n:
        return "Acquisition Corp"
    if "Inc" in n or "Incorporated" in n:
        return "Inc."
    if "Group" in n:
        return "Group"
    if "Ltd" in n or "Limited" in n:
        return "Limited"
    if "Holdings" in n or "Holding" in n:
        return "Holdings"
    return "Other"


withdrawn["Company Type"] = withdrawn[C_NAME].apply(company_type)
withdrawn["Company Type"].value_counts()

Company Type
Other               8
Limited             8
Acquisition Corp    5
Holdings            4
Technologies        3
Group               3
Inc.                1
Name: count, dtype: int64

In [6]:
withdrawn["Price_low_num"] = withdrawn[C_LOW].apply(to_num)
withdrawn["Price_high_num"] = withdrawn[C_HIGH].apply(to_num)

# Mean across the row skips NaN, so a single quoted price still gives a value
# and a fully missing range stays NaN.
withdrawn["Avg_price"] = withdrawn[["Price_low_num", "Price_high_num"]].mean(axis=1)

withdrawn["Shares_num"] = withdrawn[C_SHARES].apply(to_num)
withdrawn["Est_vol_num"] = withdrawn[C_VOL].apply(to_num)

product = withdrawn["Shares_num"] * withdrawn["Avg_price"]

# Rule: use shares * average price when it is not null, otherwise Est $ Vol.
withdrawn["Shares_offered_value"] = np.where(product.notna(), product, withdrawn["Est_vol_num"])

# Sanity check: rows with Shares = 0 fall back to Est $ Vol only because the price
# range is also missing. A zero-share row with a quoted price would produce 0 instead.
zero_shares = withdrawn["Shares_num"].fillna(0) == 0
print("rows with zero shares:", zero_shares.sum())
print("of those, also missing a price:", (zero_shares & withdrawn["Avg_price"].isna()).sum())

withdrawn[[C_NAME, "Company Type", "Shares_num", "Avg_price", "Est_vol_num", "Shares_offered_value"]]

rows with zero shares: 6
of those, also missing a price: 6


,Company,Company Type,Shares_num,Avg_price,Est_vol_num,Shares_offered_value
5,Motive Technologies (Withdrawn),Technologies,0.00,NaN,100.00,100.0000
12,Idea Tech Holding (Withdrawn),Holdings,2.00,4.50,9.00,9.0000
20,Hornbeck Offshore Services (Withdrawn),Other,0.00,NaN,100.00,100.0000
22,Coolbit Technologies Ltd. (Withdrawn),Technologies,5.00,4.50,22.50,22.5000
31,Timber Road Acquisition Corp. (Withdrawn),Acquisition Corp,20.00,10.00,200.00,200.0000
44,pan-Africa Corp. (Withdrawn),Acquisition Corp,12.50,10.00,125.00,125.0000
60,Living Homeopathy (Withdrawn),Other,3.75,5.00,19.00,18.7500
103,Nuclea Energy (Withdrawn),Other,5.60,9.00,50.40,50.4000
135,APEX Global Solutions Ltd. (WITHDRAWN),Limited,3.75,4.00,15.00,15.0000
140,Cloud Data Holdings (Withdrawn),Holdings,3.75,4.25,15.94,15.9375


In [7]:
answer_q1 = (withdrawn.groupby("Company Type")["Shares_offered_value"]
             .agg(["sum", "count"])
             .sort_values("sum", ascending=False))
print(answer_q1.round(2))
print()
print("Q1:", answer_q1.index[0], "->", round(answer_q1["sum"].iloc[0], 2), "$M")

                     sum  count
Company Type                   
Acquisition Corp  499.98      5
Inc.              351.00      1
Holdings          311.66      4
Other             290.44      8
Limited           203.85      8
Technologies      184.90      3
Group              32.50      3

Q1: Acquisition Corp -> 499.98 $M


## Q2. Median Sharpe ratio for 2025 IPOs (first 8 months)

Take the 2025 pricings list, keep IPOs priced before 1 September 2025 so that a full
trading year exists by 11 September 2026, drop 0% returns (dead or untracked tickers),
download daily OHLCV, then compute one-year growth, volatility and the Sharpe ratio.

Note on the volatility formula: the task uses `Close.rolling(30).std() * sqrt(252)`,
i.e. the standard deviation of *prices*, not of returns. That is not volatility in the
usual sense - its scale depends on the price level of the stock, so a cheap stock gets a
mechanically higher Sharpe. Keep the formula as specified, but read the result with that
in mind.

In [8]:
ipos = fetch_tables("https://www.iposcoop.com/2025-pricings/")
print(ipos.shape)
print(list(ipos.columns))
ipos.head(3)

(231, 10)
['Company', 'Symbol', 'Industry', 'Offer Date', 'Shares (millions)', 'Offer Price', '1st Day Close', 'Current Price', 'Return', 'SCOOP Rating']


,Company,Symbol,Industry,Offer Date,Shares (millions),Offer Price,1st Day Close,Current Price,Return,SCOOP Rating
0,Vine Hill Capital Investment Corp. II,VHCPU,Blank Check,12/18/2025,20.0,$10.00,$0.00,$0.00,0.00%,S/O
1,Andersen Group,ANDG,Consumer Services,12/17/2025,11.0,$16.00,$23.50,$23.79,48.69%,S/O
2,Medline Inc.,MDLN,Health Care,12/17/2025,216.0,$29.00,$41.00,$43.83,51.14%,S/O


In [9]:
C_DATE = find_col(ipos, "Offer", "Date")
C_SYMBOL = find_col(ipos, "Symbol")
C_RETURN = [c for c in ipos.columns if "return" in c.lower()][0]

ipos[C_DATE] = pd.to_datetime(ipos[C_DATE], errors="coerce")
ipos["return_num"] = (ipos[C_RETURN].astype(str)
                      .str.replace("%", "", regex=False)
                      .str.replace(",", "", regex=False)
                      .apply(to_num))

filtered = ipos[(ipos[C_DATE] < "2025-09-01") & (ipos["return_num"] != 0)].copy()
filtered = filtered.dropna(subset=[C_SYMBOL])

tickers = (filtered[C_SYMBOL].astype(str)
           .str.strip()
           .str.upper()
           .replace("", np.nan)
           .dropna()
           .unique()
           .tolist())

print(len(filtered), "rows,", len(tickers), "tickers")  # expected: ~148

146 rows, 146 tickers


In [17]:
# Batch download: one request per chunk instead of one per ticker.
# A single hanging symbol no longer blocks the whole loop.
def download_in_chunks(tickers, start, end, chunk_size=40):
    frames, missing = [], []

    for i in range(0, len(tickers), chunk_size):
        chunk = tickers[i:i + chunk_size]
        raw = yf.download(chunk, start=start, end=end, auto_adjust=True,
                          group_by="ticker", progress=False, threads=True)

        if raw is None or raw.empty:
            missing.extend(chunk)
            continue

        for ticker in chunk:
            try:
                sub = raw[ticker] if isinstance(raw.columns, pd.MultiIndex) else raw
            except KeyError:
                missing.append(ticker)
                continue

            sub = sub.dropna(how="all")
            if sub.empty:
                missing.append(ticker)
                continue

            sub = sub.reset_index()
            sub["Ticker"] = ticker
            frames.append(sub)

        print(f"{min(i + chunk_size, len(tickers))} / {len(tickers)}")

    return pd.concat(frames, ignore_index=True), missing


stocks_df, missing = download_in_chunks(tickers, "2025-01-01", "2026-09-12")
print("tickers downloaded:", stocks_df["Ticker"].nunique(), "| not found:", len(missing))

$MJID: possibly delisted; no timezone found



1 Failed download:
['MJID']: possibly delisted; no timezone found


40 / 146


$PTNM: possibly delisted; no timezone found
$SDM: possibly delisted; no timezone found
$CEPT: possibly delisted; no timezone found
$EMPG: possibly delisted; no timezone found
$AHL: possibly delisted; no timezone found
$CAEP: possibly delisted; no timezone found

6 Failed downloads:
['PTNM', 'SDM', 'CEPT', 'EMPG', 'AHL', 'CAEP']: possibly delisted; no timezone found


80 / 146


$AGH: possibly delisted; no timezone found
$TBH: possibly delisted; no timezone found

2 Failed downloads:
['AGH', 'TBH']: possibly delisted; no timezone found


120 / 146


$SKBL: possibly delisted; no timezone found
$MTSR: possibly delisted; no timezone found
$MCTR: possibly delisted; no timezone found
$EPWK: possibly delisted; no timezone found

4 Failed downloads:
['SKBL', 'MTSR', 'MCTR', 'EPWK']: possibly delisted; no timezone found


146 / 146
tickers downloaded: 133 | not found: 13


In [18]:
stocks_df = stocks_df.sort_values(["Ticker", "Date"]).reset_index(drop=True)

# groupby().transform() keeps every calculation inside a single ticker,
# so shifts never leak across the boundary between two stocks.
close_by_ticker = stocks_df.groupby("Ticker")["Close"]

stocks_df["growth_252d"] = close_by_ticker.transform(lambda s: s / s.shift(252))
stocks_df["volatility"] = close_by_ticker.transform(lambda s: s.rolling(30).std() * np.sqrt(252))
stocks_df["Sharpe"] = (stocks_df["growth_252d"] - 0.05) / stocks_df["volatility"]

snapshot = stocks_df[stocks_df["Date"].dt.strftime("%Y-%m-%d") == "2026-09-11"]
print("stocks in snapshot:", len(snapshot))
snapshot[["growth_252d", "volatility", "Sharpe"]].describe()

stocks in snapshot: 132


/usr/local/python/3.14.2/lib/python3.14/site-packages/pandas/core/nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


Price,growth_252d,volatility,Sharpe
count,130.000000,131.000000,130.000000
mean,1.057613,22.917469,inf
std,3.049007,40.995235,NaN
min,0.001005,0.000000,-0.040147
25%,0.141035,2.301309,0.012042
50%,0.594534,7.885748,0.050142
75%,1.038821,23.452319,0.133619
max,33.638270,276.424137,inf


In [19]:
print("Q2 median Sharpe:", round(snapshot["Sharpe"].median(), 3))
print("median growth_252d:", round(snapshot["growth_252d"].median(), 3),
      "| mean growth_252d:", round(snapshot["growth_252d"].mean(), 3))
print("stocks that reached the 252-day milestone:", int(snapshot["growth_252d"].notna().sum()))

Q2 median Sharpe: 0.05
median growth_252d: 0.595 | mean growth_252d: 1.058
stocks that reached the 252-day milestone: 130


## Q3. Fixed months holding strategy

Build 12 forward-growth columns (1 month = 21 trading days), then keep only the **first
trading day of each ticker** - i.e. buying at the IPO day close - and find the horizon
with the highest median.

Why the inner join matters: `min_date` gives one row per ticker, and joining it back on
(Ticker, Date) slices out exactly those IPO-day rows together with all the forward growth
values already computed on the full series. Without it the statistics would average over
every trading day instead of over the entry point.

In [20]:
for month in range(1, 13):
    days = 21 * month
    stocks_df[f"future_growth_{month}_m"] = (
        stocks_df.groupby("Ticker")["Close"]
        .transform(lambda s, days=days: s.shift(-days) / s)
    )

min_dates = stocks_df.groupby("Ticker")["Date"].min().reset_index()
first_day = min_dates.merge(stocks_df, on=["Ticker", "Date"], how="inner")
print("rows (one per ticker):", len(first_day))

growth_cols = [f"future_growth_{month}_m" for month in range(1, 13)]
desc = first_day[growth_cols].describe()
desc

rows (one per ticker): 133


,future_growth_1_m,future_growth_2_m,future_growth_3_m,future_growth_4_m,future_growth_5_m,future_growth_6_m,future_growth_7_m,future_growth_8_m,future_growth_9_m,future_growth_10_m,future_growth_11_m,future_growth_12_m
count,132.000000,131.000000,131.000000,131.000000,131.000000,131.000000,131.000000,131.000000,131.000000,131.000000,131.000000,130.000000
mean,95.751832,67.979575,51.249160,22.699034,54.244577,64.013104,59.268089,21.460472,20.010294,11.260278,8.904178,5.823784
std,1087.893336,764.388639,573.269360,247.825677,609.321778,720.716187,666.109314,234.733129,216.163486,115.714394,90.034523,55.496952
min,0.072941,0.106197,0.090147,0.063927,0.032245,0.022075,0.013229,0.015130,0.014675,0.011947,0.008681,0.004833
25%,0.710433,0.584863,0.442477,0.392759,0.413434,0.308026,0.245548,0.196227,0.175276,0.174333,0.159666,0.156935
50%,0.935351,0.892958,0.827160,0.730400,0.690909,0.726000,0.662000,0.603508,0.580294,0.533333,0.481811,0.491838
75%,1.114199,1.164977,1.129961,1.197492,1.077133,1.041222,1.018281,1.036588,1.031006,1.034765,1.106045,1.041541
max,12500.000279,8750.000196,6562.500147,2837.500063,6975.000156,8250.000184,7625.000170,2687.500060,2475.000055,1325.000030,1031.250023,633.250012


In [21]:
medians = desc.loc["50%"]
best = medians.idxmax()

print("Q3 optimal horizon:", best, "| median growth:", round(medians.max(), 4))
print()
print(pd.DataFrame({"median": medians,
                    "mean": desc.loc["mean"],
                    "count": desc.loc["count"]}).round(4))

Q3 optimal horizon: future_growth_1_m | median growth: 0.9354

                    median     mean  count
future_growth_1_m   0.9354  95.7518  132.0
future_growth_2_m   0.8930  67.9796  131.0
future_growth_3_m   0.8272  51.2492  131.0
future_growth_4_m   0.7304  22.6990  131.0
future_growth_5_m   0.6909  54.2446  131.0
future_growth_6_m   0.7260  64.0131  131.0
future_growth_7_m   0.6620  59.2681  131.0
future_growth_8_m   0.6035  21.4605  131.0
future_growth_9_m   0.5803  20.0103  131.0
future_growth_10_m  0.5333  11.2603  131.0
future_growth_11_m  0.4818   8.9042  131.0
future_growth_12_m  0.4918   5.8238  130.0


## Q4. Simple RSI-based trading strategy

A prepared parquet file with technical and macro indicators already computed.
The rule is simple: every time RSI drops below 30, invest \$1,000 and hold for 30 days.
Trade profit = 1000 x (growth_future_30d - 1).

In [15]:
import gdown

file_id = "1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-"
gdown.download(f"https://drive.google.com/uc?id={file_id}", "data.parquet", quiet=False)

df_rsi = pd.read_parquet("data.parquet", engine="pyarrow")
print(df_rsi.shape)
print([c for c in df_rsi.columns if "rsi" in c.lower() or "growth_future" in c.lower()])

Downloading...
From (original): https://drive.google.com/uc?id=1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-
From (redirected): https://drive.google.com/uc?id=1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-&confirm=t&uuid=9bcad83c-426f-400f-a50b-4a0f228b1065
To: /workspaces/stock-markets-analytics-zoomcamp-2026/02-dataframe-analysis/data.parquet
100%|██████████| 130M/130M [00:15<00:00, 8.64MB/s] 


(229932, 203)
['growth_future_30d', 'rsi', 'fastk_rsi', 'fastd_rsi', 'cdl3starsinsouth']


In [16]:
# Date may arrive as the index depending on how the file was written.
if "Date" not in df_rsi.columns:
    df_rsi = df_rsi.reset_index()
df_rsi["Date"] = pd.to_datetime(df_rsi["Date"])

RSI_COL = [c for c in df_rsi.columns if c.lower() == "rsi"][0]

selected_df = df_rsi[(df_rsi["Date"] >= "2000-01-01")
                     & (df_rsi["Date"] <= "2025-06-01")
                     & (df_rsi[RSI_COL] < 30)].copy()
print("trades:", len(selected_df))  # expected: ~5206

net_income = 1000 * (selected_df["growth_future_30d"] - 1).sum()
print("Q4 net income:", round(net_income, 2), "$ =", round(net_income / 1000, 2), "thousand $")
print("average 30-day return:", round((selected_df["growth_future_30d"].mean() - 1) * 100, 2), "%")
print("win rate:", round((selected_df["growth_future_30d"] > 1).mean() * 100, 2), "%")

trades: 5206
Q4 net income: 65805.59 $ = 65.81 thousand $
average 30-day return: 1.26 %
win rate: 55.13 %


## Q5. Predicting a positive-return IPO (exploratory)

Q3 shows that the median forward growth from the first-day close sits below 1 across
almost every horizon: buying on IPO day and holding loses money for the typical stock.
Directions worth testing:

1. **Skip the first day.** Day one prices in the hype. Entering 1-3 months later, after
   the quiet period expires and the first earnings report lands, avoids paying for it.
2. **Use lock-up expiration as the entry point.** Around day 180 insiders may sell and
   prices typically dip - a cheaper entry than the IPO close.
3. **Filter on underwriter quality.** Deals led by Goldman, Morgan Stanley or J.P. Morgan
   behave differently from \$4-6 offerings placed by small brokers; the Q1 table shows
   withdrawals concentrate in that second group.
4. **Filter on deal size.** Micro-caps (Est \$ Vol below \$50M) carry the heavy left tail;
   a size floor removes much of the loss distribution.
5. **Rank by risk-adjusted return instead of growth.** Q2 shows the mean is dragged up by
   a handful of outliers - sorting on Sharpe gives a more stable portfolio.
6. **Add a momentum filter.** Buy only IPOs trading above their offer price after a month,
   the classic winners-keep-winning effect.
7. **Replace the fixed horizon with stop-loss / take-profit.** A fixed holding period
   ignores the asymmetry of the distribution; cutting losses at -20% reshapes returns more
   than choosing between a 6- and a 9-month horizon.

Caveat: all of this needs out-of-sample validation. With roughly 130 tickers over a single
year it is easy to find a filter that fits the history and fails afterwards.